# Attributes

Yosys defines some attributes to create signals with special meaning (see [1](https://symbiyosys.readthedocs.io/en/latest/verilog.html), [2](https://yosyshq.readthedocs.io/projects/yosys/en/latest/yosys_internals/verilog.html)). cohdl_yosys provides corresponding type qualifier classes.

In [1]:
# basic setup of jupyter notebook

from example_util.jupyter_util import display_vcd

import cohdl
from cohdl import Bit, Unsigned, Signal, BitVector, Null, Port
from cohdl_yosys import YosysParams, YosysTestCase

from cohdl_yosys.formal import (
    set_default_ctx,
    always,
    cover,
)

import cohdl.std as std

cohdl.use_pretty_traceback(False)


class EmptyEntity(cohdl.Entity):
    clk = Port.input(Bit)

    def architecture(self):
        return

## Anyconst/Anyseq

Anyseq signals work pretty much like input ports of the tested entity. Yosys will try to find any sequence of states that satisfies or violates some condition. Anyconst is more restrictive and only allows stable values.

In [2]:
from cohdl_yosys.formal import Anyconst, Anyseq

def add_two_numbers(a, b):
    return a+b

class ExampleAllconstAnyseq(YosysTestCase, entity=EmptyEntity):
    _yosys_params_ = YosysParams(cover=True, bmc=True, clean_build_dir=True, quiet=True, engines="smtbmc --stbv")

    def architecture(self, dut: EmptyEntity):
        ctx = set_default_ctx(clk=std.Clock(dut.clk))

        #
        # anyconst
        #

        @std.concurrent
        def example_anyconst():
            any_a = Anyconst[Unsigned[8]](name="any_a")
            any_b = Anyconst[Unsigned[8]](name="any_b")

            # find values a and b that add to 456
            cover["reverse_sum"](add_two_numbers(any_a.resize(9), any_b.resize(9)) == 456)

        #
        # anyseq
        #

        any_seq = Anyseq[Bit](name="any_seq")

        crc = std.crc.BitwiseCrc(std.as_bitvector("111010010101"))
        crc_out = Signal[BitVector[12]](Null)

        @ctx
        def gen_crc():
            crc.update(any_seq)
            crc_out.next = crc.result()

        @std.concurrent
        def example_anyseq():
            # find some sequence of bits that results in a crc of 0xABC
            cover["crc_can_reach_abc"](crc_out.unsigned == 0xABC)

if ExampleAllconstAnyseq().test_formal_properties(return_on_error=True):
    print("formal check has passed")
    display_vcd("build/**/*.vcd", "(*.dut.)any_a:d|any_b:d|any_seq|crc_out", top_only=False)

formal check has passed
build/project_cover/engine_0/trace0.vcd


build/project_cover/engine_0/trace1.vcd


## Allconst/Allseq

Allconst and Allseq are similar to Anyconst/Anyseq. The main difference is, that assumptions are not allowed to restrict the value range of Allconst/Allseq signals. The example below passes when Anyconst is used as a qualifier but fails for Allconst. 

In [3]:
from cohdl_yosys.formal import Anyconst, Allconst, assume

class ExampleAnyAll(YosysTestCase, entity=EmptyEntity):
    _yosys_params_ = YosysParams(bmc=True, bmc_depth=2, clean_build_dir=True, quiet=True, engines="smtbmc --stbv")

    def architecture(self, dut: EmptyEntity):
        set_default_ctx(clk=std.Clock(dut.clk))

        @std.concurrent
        def example_anyconst():
            a = Qualifier[Unsigned[3]]()
            b = Qualifier[Unsigned[3]]()
            
            # not allowed when a is Allconst qualified
            assume[:](a[0])

            always["sum_works"](add_two_numbers(a, b) - a - b == 0)

Qualifier = Anyconst

if ExampleAnyAll().test_formal_properties(return_on_error=True):
    print("Anyconst: formal check has passed")

Qualifier = Allconst

if not ExampleAnyAll().test_formal_properties(return_on_error=True):
    print("Allconst: formal check has failed")


Anyconst: formal check has passed
Allconst: formal check has failed
